
Analyse exploratoire - HealthConnect Appointment Data
Semaine 4 - Track Data Analytics - AnalystLab Africa

Ce script reprend l'analyse faite pour identifier les variables
liées au taux de no-show, en vue de préparer les questions métier
et les KPI potentiels.


In [32]:
import pandas as pd
#  Chargement et aperçu général du dataset
df = pd.read_csv("HealthConnect_Appointment_Data.csv")
df 

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,HC-04996,P-0007,Male,49,45-54,Diagnostic Test,3/29/2025,4/26/2025,Saturday,Morning,28,5,1,Yes,Email,4.4,35.0,No-Show
4996,HC-04997,P-1501,Female,52,45-54,Follow-up,12/14/2025,2/12/2026,Thursday,Evening,60,6,2,Yes,WhatsApp,12.6,24.0,Attended
4997,HC-04998,P-0315,Male,59,55-64,General Consultation,3/3/2026,3/24/2026,Tuesday,Afternoon,21,5,0,No,NaN,22.8,28.0,Attended
4998,HC-04999,P-1262,Male,47,45-54,General Consultation,12/3/2025,1/6/2026,Tuesday,Morning,34,4,1,No,NaN,12.5,16.0,Attended


In [16]:
#dimmenssion
print("=== Dimensions du dataset ===")
print(df.shape)
## Types des colonnes
print("\n=== Types de colonnes ===")
print(df.dtypes)
## Verification des manquantes
print("\n=== Valeurs manquantes par colonne ===")
print(df.isnull().sum())
##Verification des doublons 
print("\n=== Doublons ===")
print("Doublons sur appointment_id :", df['appointment_id'].duplicated().sum())
print("Doublons sur lignes complètes :", df.duplicated().sum())

=== Dimensions du dataset ===
(5000, 18)

=== Types de colonnes ===
appointment_id            object
patient_id                object
gender                    object
age                        int64
age_group                 object
appointment_type          object
booking_date              object
appointment_date          object
appointment_day           object
appointment_time          object
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent             object
reminder_channel          object
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome       object
dtype: object

=== Valeurs manquantes par colonne ===
appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_ti

In [17]:
# Vérifications de cohérence
print("\n=== Répartition de appointment_outcome ===")
print(df['appointment_outcome'].value_counts())

print("\n=== reminder_sent ===")
print(df['reminder_sent'].value_counts())


=== Répartition de appointment_outcome ===
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

=== reminder_sent ===
reminder_sent
Yes    3634
No     1366
Name: count, dtype: int64


In [18]:
# Vérifie que reminder_channel n'est vide que quand reminder_sent = No
missing_no = df[df['reminder_sent'] == 'No']['reminder_channel'].isnull().sum()
total_no = (df['reminder_sent'] == 'No').sum()
print(f"\nreminder_channel manquant quand reminder_sent = No : {missing_no}/{total_no}")

missing_yes = df[(df['reminder_sent'] == 'Yes') & (df['reminder_channel'].isnull())].shape[0]
print("reminder_channel manquant quand reminder_sent = Yes (anomalie attendue) :", missing_yes)

print("\n=== Âge : min / max ===")
print(df['age'].min(), df['age'].max())

print("\n=== Incohérence previous_no_shows > previous_appointments ===")
print((df['previous_no_shows'] > df['previous_appointments']).sum())

print("\n=== Rendez-vous antérieurs à leur date de réservation (incohérence) ===")
booking = pd.to_datetime(df['booking_date'], errors='coerce')
appt = pd.to_datetime(df['appointment_date'], errors='coerce')
print((appt < booking).sum())


reminder_channel manquant quand reminder_sent = No : 1366/1366
reminder_channel manquant quand reminder_sent = Yes (anomalie attendue) : 0

=== Âge : min / max ===
18 80

=== Incohérence previous_no_shows > previous_appointments ===
0

=== Rendez-vous antérieurs à leur date de réservation (incohérence) ===
0


In [26]:
# Fonction utilitaire : taux de no-show par catégorie
def taux_no_show(colonne):
    """Calcule le taux de no-show (%) et l'effectif pour chaque
    modalité d'une colonne donnée, trié du taux le plus élevé au plus bas."""
    taux = df.groupby(colonne, observed = True)['appointment_outcome'] \
              .apply(lambda x: (x == 'No-Show').mean() * 100).round(1)
    effectifs = df.groupby(colonne, observed = True).size()
    resultat = pd.DataFrame({'taux_no_show_%': taux, 'n': effectifs})
    return resultat.sort_values('taux_no_show_%', ascending=False)

In [27]:
#Taux de no-show par variable catégorielle brute
for colonne in ['reminder_sent', 'reminder_channel', 'age_group',
                 'appointment_type', 'appointment_day', 'appointment_time']:
    print(f"\n=== Taux de No-Show par {colonne} ===")
    print(taux_no_show(colonne))


=== Taux de No-Show par reminder_sent ===
               taux_no_show_%     n
reminder_sent                      
No                       51.4  1366
Yes                      47.4  3634

=== Taux de No-Show par reminder_channel ===
                  taux_no_show_%     n
reminder_channel                      
WhatsApp                    49.8  1101
Email                       48.4   533
SMS                         45.8  2000

=== Taux de No-Show par age_group ===
           taux_no_show_%     n
age_group                      
25-34                50.7   783
55-64                50.7   800
18-24                50.2   564
35-44                48.4   819
45-54                48.0   793
65+                  45.1  1241

=== Taux de No-Show par appointment_type ===
                         taux_no_show_%     n
appointment_type                             
Follow-up                          51.2  1421
Diagnostic Test                    49.7   593
Specialist Consultation            47.4   900
G

In [28]:
#Taux de no-show par variable numérique regroupée en tranches
# Historique de no-show
df['tranche_no_shows_passes'] = pd.cut(
    df['previous_no_shows'],
    bins=[-1, 0, 1, 2, 100],
    labels=['0', '1', '2', '3+']
)
print("\n=== Taux de No-Show par historique de no-shows passés ===")
print(taux_no_show('tranche_no_shows_passes'))


=== Taux de No-Show par historique de no-shows passés ===
                         taux_no_show_%     n
tranche_no_shows_passes                      
3+                                 68.8    93
2                                  59.4   438
1                                  53.5  1548
0                                  43.5  2921


In [29]:
# Délai de réservation
df['tranche_delai_reservation'] = pd.cut(
    df['booking_lead_days'],
    bins=[-1, 3, 7, 14, 30, 100],
    labels=['0-3j', '4-7j', '8-14j', '15-30j', '30j+'])
print("\n=== Taux de No-Show par délai de réservation ===")
print(taux_no_show('tranche_delai_reservation'))


=== Taux de No-Show par délai de réservation ===
                           taux_no_show_%     n
tranche_delai_reservation                      
30j+                                 60.5  2425
15-30j                               43.2  1333
8-14j                                33.6   602
4-7j                                 30.7   322
0-3j                                 24.8   318


In [30]:
# Distance à la clinique
df['tranche_distance'] = pd.cut(
    df['distance_to_clinic_km'],
    bins=[-1, 5, 10, 20, 100],
    labels=['0-5km', '5-10km', '10-20km', '20km+'])
print("\n=== Taux de No-Show par distance à la clinique ===")
print(taux_no_show('tranche_distance'))


=== Taux de No-Show par distance à la clinique ===
                  taux_no_show_%     n
tranche_distance                      
20km+                       57.8   393
10-20km                     49.4  1669
0-5km                       46.5  1156
5-10km                      46.5  1692


In [31]:
# Temps d'attente
df['tranche_temps_attente'] = pd.cut(
    df['waiting_time_minutes'],
    bins=[-1, 15, 30, 45, 100],
    labels=['0-15min', '15-30min', '30-45min', '45min+'])
print("\n=== Taux de No-Show par temps d'attente ===")
print(taux_no_show('tranche_temps_attente'))


=== Taux de No-Show par temps d'attente ===
                       taux_no_show_%     n
tranche_temps_attente                      
15-30min                         49.2  2472
30-45min                         47.8  1260
0-15min                          47.1  1073
45min+                           46.7   135
